# Schedule validation

Independent feasibility check of the SD-CPIT schedules against the official MineLib instance files.
The check uses only the official files (`.cpit` and `.prec`) and the schedule files; it does not use any data or code of the optimisation notebooks.

For each schedule the notebook checks that:
1. every block id exists in the instance and is extracted at most once;
2. every extraction period lies within the planning horizon (`NPERIODS` of the `.cpit` file);
3. if a block is extracted in period *t*, all its predecessors are extracted in a period $\le t$;
4. the consumption of every resource in every period respects the official limits (types L, G and I);
5. the NPV, recomputed with the official objective coefficients and discount rate, matches the value reported in the paper.

The same check can be applied to the CPIT solution files distributed by MineLib (`*.sol`), which validates the procedure itself: those solutions must be feasible and reproduce the values published by MineLib.

**Files.** Download the `.cpit` files from MineLib (https://mansci-web.uai.cl/minelib/Datasets.xhtml) and place them in this folder, together with the `<instance>_solution.csv` files.
The `.prec` files are read from this folder or, if absent, from the instance folders of the repository (`../01_newman1/`, ...).
Optional: place the MineLib solution files (`<instance>_cpit_*.sol`) in this folder to validate them as well.

In [1]:
import glob
import os
import time

import numpy as np
import pandas as pd

# Instance folders of the repository (used to find the .prec files if they are not in this folder)
FOLDERS = {"newman1": "01_newman1", "zuck_small": "02_zuck_small", "kd": "03_kd", "zuck_medium": "04_zuck_medium",
           "p4hd": "05_p4hd", "marvin": "06_marvin", "w23": "07_w23", "zuck_large": "08_zuck_large", "sm2": "09_sm2",
           "mclaughlin_limit": "10_mclaughlin_limit", "mclaughlin": "11_mclaughlin"}

# NPV reported in Table 2 of the paper (SD-CPIT)
REPORTED = {"newman1": 24_176_865, "zuck_small": 847_050_785, "kd": 409_106_516, "zuck_medium": 669_502_198,
            "p4hd": 247_406_799, "marvin": 856_217_864, "w23": 399_973_326, "zuck_large": 57_324_534,
            "sm2": 1_647_381_288, "mclaughlin_limit": 1_078_669_389, "mclaughlin": 1_078_712_761}

TOL = 1e-6  # relative tolerance for the resource limits

def find(filename, instance):
    """Return the path of a file in this folder or in the instance folder of the repository."""
    for path in (filename, os.path.join("..", FOLDERS[instance], filename)):
        if os.path.exists(path):
            return path
    return None

## Readers

In [2]:
def read_cpit(path):
    """Read a MineLib .cpit file: horizon, discount rate, resource limits, objective and resource coefficients."""
    with open(path) as f:
        lines = [l.strip() for l in f.read().splitlines()]
    header, i = {}, 0
    while not lines[i].startswith("RESOURCE_CONSTRAINT_LIMITS"):
        if ":" in lines[i]:
            k, v = lines[i].split(":", 1)
            header[k.strip()] = v.strip()
        i += 1
    n = int(header["NBLOCKS"])
    T = int(header["NPERIODS"])
    nr = int(header["NRESOURCE_SIDE_CONSTRAINTS"])
    rate = float(header["DISCOUNT_RATE"])

    lb = np.full((nr, T), -np.inf)
    ub = np.full((nr, T), np.inf)
    i += 1
    while not lines[i].startswith("OBJECTIVE_FUNCTION"):
        p = lines[i].split()
        if p:
            r, t, kind = int(p[0]), int(p[1]), p[2]
            if kind == "L":
                ub[r, t] = float(p[3])
            elif kind == "G":
                lb[r, t] = float(p[3])
            else:  # "I": interval
                lb[r, t], ub[r, t] = float(p[3]), float(p[4])
        i += 1

    ir = lines.index("RESOURCE_CONSTRAINT_COEFFICIENTS:")
    obj = np.zeros(n)
    for l in lines[i + 1:ir]:
        p = l.split()
        if len(p) == 2:
            obj[int(p[0])] = float(p[1])
    res = np.zeros((nr, n))
    for l in lines[ir + 1:]:
        p = l.split()
        if len(p) == 3:
            res[int(p[1]), int(p[0])] = float(p[2])
    return n, T, rate, lb, ub, obj, res

def read_schedule(path, n):
    """Return the extraction period of every block (-1 = not extracted) and the list of errors.

    CSV files (this repository): columns "id" and "period" (period < 0 or empty = not extracted).
    MineLib .sol files: one line per variable, "block destination period value".
    """
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path, usecols=["id", "period"])
        p = pd.to_numeric(df["period"], errors="coerce")
        mask = p.notna() & (p >= 0)
        b = df.loc[mask, "id"].astype(np.int64).to_numpy()
        t = p[mask].astype(np.int64).to_numpy()
    else:
        data = np.loadtxt(path, ndmin=2)
        data = data[data[:, 3] > 0.5]
        b, t = data[:, 0].astype(np.int64), data[:, 2].astype(np.int64)

    errors = []
    unknown = (b < 0) | (b >= n)
    if unknown.any():
        errors.append(f"{int(unknown.sum())} unknown block ids")
    b, t = b[~unknown], t[~unknown]
    repeated = np.bincount(b, minlength=n) > 1
    if repeated.any():
        errors.append(f"{int(repeated.sum())} blocks extracted more than once")
    period = np.full(n, -1, dtype=np.int64)
    period[b] = t
    return period, errors

## Checks

In [3]:
def check(instance, schedule_path):
    """Check one schedule and return a dictionary with the results."""
    start = time.time()
    n, T, rate, lb, ub, obj, res = read_cpit(find(instance + ".cpit", instance))
    period, errors = read_schedule(schedule_path, n)
    ext = period >= 0

    # 2. Planning horizon
    late = int((period >= T).sum())
    if late:
        errors.append(f"{late} blocks after the planning horizon")

    # 3. Precedence (streamed line by line, since some .prec files are large)
    arcs = violated = 0
    with open(find(instance + ".prec", instance)) as f:
        for line in f:
            q = line.split()
            if len(q) < 3:
                continue
            tb = period[int(q[0])]
            preds = np.array(q[2:2 + int(q[1])], dtype=np.int64)
            arcs += len(preds)
            if tb >= 0:
                tp = period[preds]
                violated += int(((tp < 0) | (tp > tb)).sum())
    if violated:
        errors.append(f"{violated} precedence arcs violated")

    # 4. Resources: consumption per resource and period
    t_ok = ext & (period < T)
    worst = 0.0
    for r in range(res.shape[0]):
        use = np.bincount(period[t_ok], weights=res[r, t_ok], minlength=T)[:T]
        hi = np.where(np.isfinite(ub[r]), ub[r] + TOL * np.maximum(1, np.abs(ub[r])), np.inf)
        lo = np.where(np.isfinite(lb[r]), lb[r] - TOL * np.maximum(1, np.abs(lb[r])), -np.inf)
        bad = np.where((use > hi) | (use < lo))[0]
        for t in bad:
            errors.append(f"resource {r}, period {t}: use {use[t]:,.2f} outside [{lb[r, t]:,.2f}, {ub[r, t]:,.2f}]")
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(np.isfinite(ub[r]) & (ub[r] > 0), use / ub[r], 0)
        worst = max(worst, float(ratio.max()))

    # 5. NPV with the official objective coefficients and discount rate
    npv = float((obj[t_ok] / (1 + rate) ** period[t_ok]).sum())

    return {"instance": instance, "file": os.path.basename(schedule_path), "horizon": T,
            "periods used": int(period.max()) + 1 if ext.any() else 0, "blocks extracted": int(ext.sum()),
            "precedence arcs": arcs, "violated arcs": violated, "max use of a limit (%)": round(100 * worst, 4),
            "NPV recomputed": round(npv), "status": "FEASIBLE" if not errors else "INFEASIBLE",
            "errors": "; ".join(errors), "time (s)": round(time.time() - start, 1)}

## SD-CPIT schedules

In [4]:
rows = []
for instance in FOLDERS:
    path = find(instance + "_solution.csv", instance)
    if path is None or find(instance + ".cpit", instance) is None or find(instance + ".prec", instance) is None:
        print(f"{instance}: files not found, skipped")
        continue
    row = check(instance, path)
    row["NPV reported"] = REPORTED[instance]
    row["difference"] = row["NPV recomputed"] - REPORTED[instance]
    rows.append(row)
    print(f"{instance}: {row['status']}, NPV {row['NPV recomputed']:,}, {row['time (s)']} s")

sdcpit = pd.DataFrame(rows)
sdcpit.to_csv("schedule_validation_sdcpit.csv", index=False)
sdcpit

newman1: FEASIBLE, NPV 24,176,865, 0.0 s
zuck_small: FEASIBLE, NPV 847,050,785, 0.1 s
kd: FEASIBLE, NPV 409,106,516, 0.1 s
zuck_medium: FEASIBLE, NPV 669,502,198, 0.4 s
p4hd: FEASIBLE, NPV 247,406,799, 0.3 s
marvin: FEASIBLE, NPV 856,217,864, 0.3 s
w23: FEASIBLE, NPV 399,973,326, 0.6 s
zuck_large: FEASIBLE, NPV 57,324,534, 1.1 s
sm2: FEASIBLE, NPV 1,647,381,288, 0.4 s
mclaughlin_limit: FEASIBLE, NPV 1,078,669,389, 1.2 s
mclaughlin: FEASIBLE, NPV 1,078,712,761, 13.2 s


,instance,file,horizon,periods used,blocks extracted,precedence arcs,violated arcs,max use of a limit (%),NPV recomputed,status,errors,time (s),NPV reported,difference
0,newman1,newman1_solution.csv,6,3,1059,3922,0,99.9624,24176865,FEASIBLE,,0.0,24176865,0
1,zuck_small,zuck_small_solution.csv,20,17,9399,145640,0,99.9999,847050785,FEASIBLE,,0.1,847050785,0
2,kd,kd_solution.csv,12,10,12154,219778,0,99.9996,409106516,FEASIBLE,,0.1,409106516,0
3,zuck_medium,zuck_medium_solution.csv,15,14,27387,1271207,0,99.9998,669502198,FEASIBLE,,0.4,669502198,0
4,p4hd,p4hd_solution.csv,10,6,28844,738609,0,100.0000,247406799,FEASIBLE,,0.3,247406799,0
5,marvin,marvin_solution.csv,20,16,8516,650631,0,99.9995,856217864,FEASIBLE,,0.3,856217864,0
6,w23,w23_solution.csv,12,11,49061,764786,0,99.9729,399973326,FEASIBLE,,0.6,399973326,0
7,zuck_large,zuck_large_solution.csv,30,25,96821,1053105,0,100.0000,57324534,FEASIBLE,,1.1,57324534,0
8,sm2,sm2_solution.csv,30,22,18388,96642,0,100.0000,1647381288,FEASIBLE,,0.4,1647381288,0
9,mclaughlin_limit,mclaughlin_limit_solution.csv,15,10,110225,3035483,0,100.0000,1078669389,FEASIBLE,,1.2,1078669389,0


## MineLib solution files (validation of the procedure)

Only the `.sol` files present in this folder are checked. Fractional (LP) solution files are not CPIT schedules and should not be placed here.

In [5]:
rows = []
for instance in FOLDERS:
    if find(instance + ".cpit", instance) is None or find(instance + ".prec", instance) is None:
        continue
    for path in sorted(glob.glob(instance + "_cpit_*.sol")):
        row = check(instance, path)
        rows.append(row)
        print(f"{os.path.basename(path)}: {row['status']}, NPV {row['NPV recomputed']:,}")

minelib = pd.DataFrame(rows)
if len(minelib):
    minelib.to_csv("schedule_validation_minelib.csv", index=False)
minelib

newman1_cpit_gmunoz120723.sol: FEASIBLE, NPV 23,483,671
zuck_small_cpit_gmunoz120723.sol: FEASIBLE, NPV 788,652,600
kd_cpit_gmunoz120723.sol: FEASIBLE, NPV 396,858,192
zuck_medium_cpit_gmunoz120723.sol: FEASIBLE, NPV 615,411,415
p4hd_cpit_gmunoz120723.sol: FEASIBLE, NPV 246,138,696
marvin_cpit_gmunoz120723.sol: FEASIBLE, NPV 820,726,048
w23_cpit_gmunoz120723.sol: FEASIBLE, NPV 392,226,063
zuck_large_cpit_gmunoz120723.sol: FEASIBLE, NPV 56,777,190
sm2_cpit_gmunoz120723.sol: FEASIBLE, NPV 1,645,242,774
mclaughlin_limit_cpit_gmunoz120723.sol: FEASIBLE, NPV 1,073,327,197
mclaughlin_cpit_gmunoz120723.sol: FEASIBLE, NPV 1,073,530,279


,instance,file,horizon,periods used,blocks extracted,precedence arcs,violated arcs,max use of a limit (%),NPV recomputed,status,errors,time (s)
0,newman1,newman1_cpit_gmunoz120723.sol,6,4,1059,3922,0,99.9973,23483671,FEASIBLE,,0.0
1,zuck_small,zuck_small_cpit_gmunoz120723.sol,20,17,9380,145640,0,99.9996,788652600,FEASIBLE,,0.1
2,kd,kd_cpit_gmunoz120723.sol,12,10,12154,219778,0,99.9996,396858192,FEASIBLE,,0.1
3,zuck_medium,zuck_medium_cpit_gmunoz120723.sol,15,14,27387,1271207,0,99.9999,615411415,FEASIBLE,,0.4
4,p4hd,p4hd_cpit_gmunoz120723.sol,10,6,28844,738609,0,99.9984,246138696,FEASIBLE,,0.3
5,marvin,marvin_cpit_gmunoz120723.sol,20,16,8516,650631,0,99.9868,820726048,FEASIBLE,,0.2
6,w23,w23_cpit_gmunoz120723.sol,12,11,49061,764786,0,99.9729,392226063,FEASIBLE,,0.4
7,zuck_large,zuck_large_cpit_gmunoz120723.sol,30,25,96821,1053105,0,99.9992,56777190,FEASIBLE,,0.8
8,sm2,sm2_cpit_gmunoz120723.sol,30,22,18388,96642,0,99.9989,1645242774,FEASIBLE,,0.4
9,mclaughlin_limit,mclaughlin_limit_cpit_gmunoz120723.sol,15,10,110225,3035483,0,99.9994,1073327197,FEASIBLE,,1.2


In [6]:
print(f"SD-CPIT schedules checked: {len(sdcpit)}, feasible: {(sdcpit['status'] == 'FEASIBLE').sum()}")
if len(minelib):
    print(f"MineLib solutions checked: {len(minelib)}, feasible: {(minelib['status'] == 'FEASIBLE').sum()}")

SD-CPIT schedules checked: 11, feasible: 11
MineLib solutions checked: 11, feasible: 11
